-------->  LIBRARIES && Load Dataset

In [1]:
import pandas as pd
import numpy as np

data = pd.read_csv("Earthquakes_Dataset.csv")

print("First 5 Rows:")
print(data.head())

print("\nDataset Shape:", data.shape)

First 5 Rows:
                             place   mag magType        type  \
0         113 km W of Petrolia, CA  2.56      md  earthquake   
1               9 km N of Taft, CA  1.54      ml  earthquake   
2        58 km WNW of Petrolia, CA  2.84      md  earthquake   
3  24 km NNW of Searles Valley, CA  1.49      ml  earthquake   
4            12 km WNW of Anza, CA  0.56      ml  earthquake   

                      time   longitude   latitude  depth_km  sig net   nst  \
0  2025-02-11 14:34:38.190 -125.614334  40.216499      4.95  101  nc  14.0   
1  2025-02-11 14:28:19.490 -119.456833  35.225167     18.24   36  ci  31.0   
2  2025-02-11 14:24:27.270 -124.964500  40.401165      4.20  124  nc  30.0   
3  2025-02-11 14:24:22.730 -117.515663  35.967335      4.01   34  ci  19.0   
4  2025-02-11 14:12:30.720 -116.803167  33.584500      8.69    5  ci  25.0   

      dmin   rms    gap  
0  0.97600  0.12  325.0  
1  0.08479  0.20  116.0  
2  0.48850  0.37  274.0  
3  0.11100  0.16   83.0  
4 

-------->     Check Missing Values

In [2]:
print("Missing Values in Each Column:")
print(data.isnull().sum())

Missing Values in Each Column:
place        0
mag          0
magType      0
type         0
time         0
longitude    0
latitude     0
depth_km     0
sig          0
net          0
nst          0
dmin         0
rms          0
gap          0
dtype: int64


------->  Handle Missing Values

In [3]:
# Fill numerical columns with median
num_cols = data.select_dtypes(include=np.number).columns
data[num_cols] = data[num_cols].fillna(data[num_cols].median())

# Fill categorical columns with mode
cat_cols = data.select_dtypes(include='object').columns

for col in cat_cols:
    data[col].fillna(data[col].mode()[0], inplace=True)

print("Missing values handled")

Missing values handled


----->  Convert Date/Time Feature

In [4]:
if 'time' in data.columns:

    data['time'] = pd.to_datetime(data['time'], errors='coerce')

    data['year'] = data['time'].dt.year
    data['month'] = data['time'].dt.month
    data['day'] = data['time'].dt.day

print(data[['year','month','day']].head())

     year  month   day
0  2025.0    2.0  11.0
1  2025.0    2.0  11.0
2  2025.0    2.0  11.0
3  2025.0    2.0  11.0
4  2025.0    2.0  11.0


------->   Feature Engineering

In [5]:
# Create magnitude category
if 'mag' in data.columns:
    data['magnitude_level'] = pd.cut(
        data['mag'],
        bins=[0,4,6,10],
        labels=['Low','Moderate','High']
    )

print(data[['mag','magnitude_level']].head())

    mag magnitude_level
0  2.56             Low
1  1.54             Low
2  2.84             Low
3  1.49             Low
4  0.56             Low


------>   Normalize Numerical Features

In [6]:
from sklearn.preprocessing import MinMaxScaler

scaler = MinMaxScaler()

if 'mag' in data.columns and 'depth' in data.columns:
    data[['mag','depth']] = scaler.fit_transform(data[['mag','depth']])

print("Normalization completed")

Normalization completed


------>   Remove Duplicate Data

In [7]:
before = data.shape[0]

data = data.drop_duplicates()

after = data.shape[0]

print("Duplicates removed:", before-after)

Duplicates removed: 13928


-------->   Outlier Detection (IQR Method)

In [8]:
if 'mag' in data.columns:
    Q1 = data['mag'].quantile(0.25)
    Q3 = data['mag'].quantile(0.75)
    IQR = Q3 - Q1

    lower = Q1 - 1.5 * IQR
    upper = Q3 + 1.5 * IQR

    data = data[(data['mag'] >= lower) & (data['mag'] <= upper)]

print("Outliers removed")

Outliers removed


-------->  Save Preprocessed Dataset

In [9]:
data.to_csv("preprocessed_earthquake_data.csv", index=False)

print("Preprocessed dataset saved successfully")

Preprocessed dataset saved successfully


-------->  Final dataset shape

In [10]:
print("Final Dataset Shape:", data.shape)
data.head()

Final Dataset Shape: (7585, 18)


,place,mag,magType,type,time,longitude,latitude,depth_km,sig,net,nst,dmin,rms,gap,year,month,day,magnitude_level
0,"113 km W of Petrolia, CA",2.56,md,earthquake,2025-02-11 14:34:38.190,-125.614334,40.216499,4.95,101,nc,14.0,0.97600,0.12,325.0,2025.0,2.0,11.0,Low
1,"9 km N of Taft, CA",1.54,ml,earthquake,2025-02-11 14:28:19.490,-119.456833,35.225167,18.24,36,ci,31.0,0.08479,0.20,116.0,2025.0,2.0,11.0,Low
2,"58 km WNW of Petrolia, CA",2.84,md,earthquake,2025-02-11 14:24:27.270,-124.964500,40.401165,4.20,124,nc,30.0,0.48850,0.37,274.0,2025.0,2.0,11.0,Low
3,"24 km NNW of Searles Valley, CA",1.49,ml,earthquake,2025-02-11 14:24:22.730,-117.515663,35.967335,4.01,34,ci,19.0,0.11100,0.16,83.0,2025.0,2.0,11.0,Low
4,"12 km WNW of Anza, CA",0.56,ml,earthquake,2025-02-11 14:12:30.720,-116.803167,33.584500,8.69,5,ci,25.0,0.05820,0.16,69.0,2025.0,2.0,11.0,Low
